# LeetCode #130 — Surrounded Regions: đánh dấu từ biên vào trong

**Mức độ:** Medium · **Chủ đề:** Array, Depth-First Search, Breadth-First Search, Union Find, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def solve(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
```

Mục tiêu của notebook:

1. Đổi góc nhìn: thay vì hỏi "vùng nào bị bao", hỏi "vùng nào **thoát** được ra biên".
2. Flood fill (loang) từ các ô `'O'` trên **biên**, đánh dấu tạm `'#'`.
3. Ba pha: đánh dấu → lật `'O'` còn lại thành `'X'` → khôi phục `'#'` thành `'O'`.
4. Cài đặt bằng BFS (tránh đệ quy sâu) và bằng Union-Find.

# 1. Bài toán là gì?

Cho bảng `m × n` gồm `'X'` và `'O'`. Một **vùng** là tập các ô `'O'` nối nhau theo 4 hướng. Vùng bị **bao quanh** nếu **không** có ô nào của nó nằm trên **biên** bảng (khi đó nó bị các ô `'X'` vây kín).

Lật mọi `'O'` của các vùng bị bao quanh thành `'X'`, **tại chỗ**.

**Constraints (theo đề):**

- `m == board.length`, `n == board[i].length`
- `1 <= m, n <= 200`
- `board[i][j]` là `'X'` hoặc `'O'`.

---

## Ví dụ

```text
X X X X        X X X X
X O O X   ->   X X X X
X X O X        X X X X
X O X X        X O X X      ('O' ở hàng cuối nằm trên biên -> giữ)

[["X"]]  ->  [["X"]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python
2. Duyệt theo chiều sâu (DFS — Depth-First Search)
3. Duyệt theo chiều rộng (BFS — Breadth-First Search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [1]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [2]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [4]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [5]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [6]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [7]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [8]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.3 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [9]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [10]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [11]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.4 Áp dụng vào bài này

Vùng `'O'` **không** bị bao là vùng chạm **biên**. Ta **DFS/BFS** (tô loang) từ mọi ô `'O'` trên biên để đánh dấu tạm các ô "an toàn", rồi duyệt **ma trận** một lượt: `'O'` còn lại bị bao → `'X'`, ô đã đánh dấu → `'O'`.

# 3. Nền tảng

In [12]:
from typing import List
from collections import deque
import random, copy


def show(board):
    for row in board:
        print("  " + " ".join(row))


def parse(rows):
    return [list(r) for r in rows]

EX = parse(["XXXX", "XOOX", "XXOX", "XOXX"])
show(EX)

  X X X X
  X O O X
  X X O X
  X O X X


# 4. Cách ngây thơ: với mỗi vùng, kiểm tra có chạm biên không

Tìm từng vùng bằng BFS; nếu không ô nào của vùng nằm trên biên thì lật cả vùng. Cách này **đúng** và cũng `O(m·n)` nếu đánh dấu đã thăm — nhưng phải lưu danh sách ô của từng vùng.

In [13]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def solve_by_regions(board: List[List[str]]) -> None:
    m, n = len(board), len(board[0])
    seen = [[False] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if board[r][c] != "O" or seen[r][c]:
                continue
            region, touches = [], False
            q = deque([(r, c)])
            seen[r][c] = True
            while q:
                x, y = q.popleft()
                region.append((x, y))
                if x in (0, m - 1) or y in (0, n - 1):
                    touches = True
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and not seen[nx][ny] and board[nx][ny] == "O":
                        seen[nx][ny] = True
                        q.append((nx, ny))
            if not touches:
                for x, y in region:
                    board[x][y] = "X"

b = copy.deepcopy(EX); solve_by_regions(b); show(b)

  X X X X
  X X X X
  X X X X
  X O X X


# 5. Ý tưởng chính: loang từ biên

Một ô `'O'` **không** bị lật ⇔ nó nối được (qua các ô `'O'`) tới một ô `'O'` trên biên.

Vậy chỉ cần **loang từ mọi ô `'O'` trên biên**, đánh dấu các ô "an toàn" bằng ký tự tạm `'#'`. Sau đó:

```text
'O' (không được đánh dấu)  ->  'X'      # bị bao
'#'                        ->  'O'      # an toàn, khôi phục
```

Dùng chính `board` để đánh dấu → không cần mảng `seen`.

In [14]:
def solve_border(board: List[List[str]]) -> None:
    m, n = len(board), len(board[0])
    q = deque()
    for r in range(m):
        for c in (0, n - 1):
            if board[r][c] == "O":
                board[r][c] = "#"; q.append((r, c))
    for c in range(n):
        for r in (0, m - 1):
            if board[r][c] == "O":
                board[r][c] = "#"; q.append((r, c))
    while q:
        x, y = q.popleft()
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "O":
                board[nx][ny] = "#"
                q.append((nx, ny))
    for r in range(m):
        for c in range(n):
            if board[r][c] == "O":
                board[r][c] = "X"
            elif board[r][c] == "#":
                board[r][c] = "O"

b = copy.deepcopy(EX); solve_border(b); show(b)

  X X X X
  X X X X
  X X X X
  X O X X


Đánh dấu `'#'` **ngay khi đưa vào hàng đợi** (không phải khi lấy ra) để mỗi ô vào hàng đợi đúng một lần.

# 6. Mô phỏng

In [15]:
def trace_border(rows):
    board = parse(rows)
    m, n = len(board), len(board[0])
    q = deque()
    for r in range(m):
        for c in range(n):
            if (r in (0, m - 1) or c in (0, n - 1)) and board[r][c] == "O":
                board[r][c] = "#"; q.append((r, c))
    print("Pha 1a — đánh dấu 'O' trên biên:"); show(board)
    while q:
        x, y = q.popleft()
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "O":
                board[nx][ny] = "#"; q.append((nx, ny))
    print("Pha 1b — loang vào trong:"); show(board)
    for r in range(m):
        for c in range(n):
            board[r][c] = {"O": "X", "#": "O"}.get(board[r][c], board[r][c])
    print("Pha 2 — lật 'O' còn lại, khôi phục '#':"); show(board)
    return board

_ = trace_border(["XXXXX", "XOOXO", "XXOXO", "XOXOX", "XOXXX"])

Pha 1a — đánh dấu 'O' trên biên:
  X X X X X
  X O O X #
  X X O X #
  X O X O X
  X # X X X
Pha 1b — loang vào trong:
  X X X X X
  X O O X #
  X X O X #
  X # X O X
  X # X X X
Pha 2 — lật 'O' còn lại, khôi phục '#':
  X X X X X
  X X X X O
  X X X X O
  X O X X X
  X O X X X


Ở ví dụ này: vùng `(1,1),(1,2),(2,2)` bị bao → lật. Cột cuối có `'O'` ở `(1,4),(2,4)` trên biên → giữ. `(4,1)` và `(3,1)` nối biên dưới → giữ. `(3,3)` bị bao → lật.

# 7. Cách 3: Union-Find với một nút "biên" ảo

Thêm một nút ảo `EDGE`. Hợp mọi ô `'O'` trên biên với `EDGE`; hợp các ô `'O'` kề nhau. Cuối cùng, ô `'O'` nào **không** cùng thành phần với `EDGE` thì lật.

In [16]:
def solve_union_find(board: List[List[str]]) -> None:
    m, n = len(board), len(board[0])
    EDGE = m * n
    parent = list(range(m * n + 1))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for r in range(m):
        for c in range(n):
            if board[r][c] != "O":
                continue
            idx = r * n + c
            if r in (0, m - 1) or c in (0, n - 1):
                union(idx, EDGE)
            if r + 1 < m and board[r + 1][c] == "O":
                union(idx, idx + n)
            if c + 1 < n and board[r][c + 1] == "O":
                union(idx, idx + 1)
    root_edge = find(EDGE)
    for r in range(m):
        for c in range(n):
            if board[r][c] == "O" and find(r * n + c) != root_edge:
                board[r][c] = "X"

b = copy.deepcopy(EX); solve_union_find(b); show(b)

  X X X X


  X X X X
  X X X X
  X O X X


Chỉ cần hợp với ô **dưới** và **phải** — cạnh với ô trên/trái đã được xét khi duyệt ô đó.

# 8. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def solve(self, board: List[List[str]]) -> None:
        m, n = len(board), len(board[0])
        q = deque()
        for r in range(m):
            for c in range(n):
                if (r in (0, m - 1) or c in (0, n - 1)) and board[r][c] == "O":
                    board[r][c] = "#"
                    q.append((r, c))
        while q:
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "O":
                    board[nx][ny] = "#"
                    q.append((nx, ny))
        for r in range(m):
            for c in range(n):
                if board[r][c] == "O":
                    board[r][c] = "X"
                elif board[r][c] == "#":
                    board[r][c] = "O"

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS")


def run(f, rows):
    b = parse(rows) if isinstance(rows[0], str) else copy.deepcopy(rows)
    assert f(b) is None
    return ["".join(r) for r in b]


sol = Solution()
check("Ví dụ 1", run(sol.solve, ["XXXX", "XOOX", "XXOX", "XOXX"]), ["XXXX", "XXXX", "XXXX", "XOXX"])
check("Ví dụ 2", run(sol.solve, ["X"]), ["X"])
check("Toàn O", run(sol.solve, ["OOO", "OOO", "OOO"]), ["OOO", "OOO", "OOO"])
check("Một hàng", run(sol.solve, ["OXO"]), ["OXO"])
check("O ở giữa", run(sol.solve, ["XXX", "XOX", "XXX"]), ["XXX", "XXX", "XXX"])

random.seed(130)
for _ in range(2000):
    m, n = random.randint(1, 7), random.randint(1, 7)
    board = [[random.choice("XO") for _ in range(n)] for _ in range(m)]
    expected = run(solve_by_regions, board)
    assert run(sol.solve, board) == expected == run(solve_union_find, board)
big = [["O"] * 200 for _ in range(200)]
check("200x200 toàn O", run(sol.solve, big) == ["O" * 200] * 200, True)
print("2000 bảng ngẫu nhiên: 3 cách khớp nhau")

Ví dụ 1            PASS


Ví dụ 2            PASS
Toàn O             PASS
Một hàng           PASS
O ở giữa           PASS


200x200 toàn O     PASS
2000 bảng ngẫu nhiên: 3 cách khớp nhau


Bảng `200 × 200` toàn `'O'` là trường hợp làm DFS đệ quy **tràn ngăn xếp** (độ sâu tới 40 000) — BFS với hàng đợi không bị.

# 9. Vì sao đúng? Độ phức tạp?

- Ô `'O'` được đánh dấu `'#'` ⇔ có đường đi qua các ô `'O'` từ một ô `'O'` trên biên tới nó (BFS từ nhiều nguồn tìm đúng tập ô đạt được).
- Vùng chạm biên ⇔ mọi ô của nó đều đạt được từ biên (vì vùng liên thông). Vùng không chạm biên ⇔ không ô nào đạt được.
- Pha cuối lật đúng các ô của vùng bị bao, khôi phục các ô an toàn.

```text
Time:  O(m · n) — mỗi ô vào hàng đợi tối đa một lần
Space: O(m · n) xấu nhất cho hàng đợi (không cần mảng seen)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Lật ngay khi gặp `'O'` không nằm trên biên

In [19]:
def wrong_flip_inner(board):
    m, n = len(board), len(board[0])
    for r in range(1, m - 1):
        for c in range(1, n - 1):
            if board[r][c] == "O":
                board[r][c] = "X"                  # SAI: có thể nối ra biên

print("Sai :", run(wrong_flip_inner, ["XXXX", "XOOX", "XXOX", "XOOX", "XXOX"]))
print("Đúng:", run(sol.solve, ["XXXX", "XOOX", "XXOX", "XOOX", "XXOX"]))

Sai : ['XXXX', 'XXXX', 'XXXX', 'XXXX', 'XXOX']
Đúng: ['XXXX', 'XOOX', 'XXOX', 'XOOX', 'XXOX']


Vùng bên trong nối ra `(4, 2)` trên biên dưới → không bị bao.

## Lỗi 2: DFS đệ quy trên bảng lớn

In [20]:
import sys

def solve_dfs_recursive(board):
    m, n = len(board), len(board[0])
    def dfs(x, y):
        if not (0 <= x < m and 0 <= y < n) or board[x][y] != "O":
            return
        board[x][y] = "#"
        for dx, dy in DIRS:
            dfs(x + dx, y + dy)
    for r in range(m):
        for c in range(n):
            if (r in (0, m - 1) or c in (0, n - 1)):
                dfs(r, c)
    for r in range(m):
        for c in range(n):
            board[r][c] = {"O": "X", "#": "O"}.get(board[r][c], board[r][c])

print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    solve_dfs_recursive([["O"] * 60 for _ in range(60)])
    print("60x60: chạy được")
except RecursionError:
    print("60x60 toàn 'O': RecursionError")

Giới hạn đệ quy mặc định: 1000
60x60 toàn 'O': RecursionError


DFS đệ quy **đúng về logic**, nhưng với bảng `200 × 200` có thể sâu tới hàng chục nghìn tầng. Trong Python nên dùng BFS hoặc stack tự quản.

## Lỗi 3: Đánh dấu khi lấy ra khỏi hàng đợi (thay vì khi đưa vào)

Một ô có thể được đưa vào hàng đợi nhiều lần (từ nhiều hàng xóm) trước khi được đánh dấu → tốn thời gian và bộ nhớ, xấu nhất gấp 4 lần.

## Lỗi 4: Quên khôi phục `'#'` về `'O'`

Bảng kết quả chứa ký tự lạ `'#'`.

# 11. Mẹo nhận diện pattern

**"Vùng bị bao / không thoát được / không chạm biên"** → **loang ngược từ biên** (multi-source BFS/DFS).

Bài liên quan: **#200 Number of Islands**, **#1020 Number of Enclaves** (đếm ô đất không ra được biên), **#1254 Number of Closed Islands**, **#417 Pacific Atlantic Water Flow** (loang từ hai biên), **#542 01 Matrix** (BFS nhiều nguồn).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Áp dụng thuật toán cho:

```text
O X X O X
X O O X O
X O X O X
O X O O O
X X O X O
```

## Bài 2 — Điền chỗ trống

In [21]:
def exercise_2_solve(board: List[List[str]]) -> None:
    m, n = len(board), len(board[0])
    q = deque()
    # TODO: pha 1a — đánh dấu 'O' trên biên và đưa vào q
    # TODO: pha 1b — BFS loang
    # TODO: pha 2 — 'O' -> 'X', '#' -> 'O'


# b = parse(["XXXX", "XOOX", "XXOX", "XOXX"]); exercise_2_solve(b); assert ["".join(r) for r in b] == ["XXXX", "XXXX", "XXXX", "XOXX"]
# b = parse(["X"]); exercise_2_solve(b); assert b == [["X"]]

## Bài 3 — Biến thể: Number of Enclaves (LeetCode #1020)

Lưới `0` (nước) / `1` (đất). Đếm số ô đất **không thể** đi ra biên (đi 4 hướng trên đất).

```python
num_enclaves([[0,0,0,0],[1,0,1,0],[0,1,1,0],[0,0,0,0]]) -> 3
num_enclaves([[0,1,1,0],[0,0,1,0],[0,0,1,0],[0,0,0,0]]) -> 0
```

In [22]:
def exercise_3_num_enclaves(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_3_num_enclaves([[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]) == 3
# assert exercise_3_num_enclaves([[0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]]) == 0

## Bài 4 — Biến thể: Number of Closed Islands (LeetCode #1254)

Lưới `0` (đất) / `1` (nước) — chú ý ngược với Bài 3. Đếm số **đảo** (vùng đất) **không** chạm biên.

```python
closed_island([[1,1,1,1,1,1,1,0],[1,0,0,0,0,1,1,0],[1,0,1,0,1,1,1,0],[1,0,0,0,0,1,0,1],[1,1,1,1,1,1,1,0]]) -> 2
```

In [23]:
def exercise_4_closed_island(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_closed_island([[1,1,1,1,1,1,1,0],[1,0,0,0,0,1,1,0],[1,0,1,0,1,1,1,0],[1,0,0,0,0,1,0,1],[1,1,1,1,1,1,1,0]]) == 2
# assert exercise_4_closed_island([[0,0,1,0,0],[0,1,0,1,0],[0,1,1,1,0]]) == 1

## Bài 5 — Mini challenge: Pacific Atlantic Water Flow (LeetCode #417)

Lưới độ cao `heights`. Thái Bình Dương chạm biên **trên và trái**, Đại Tây Dương chạm biên **dưới và phải**. Nước chảy từ ô sang ô kề có độ cao **không lớn hơn**. Trả về các ô mà nước có thể chảy ra **cả hai** đại dương.

Gợi ý nhanh: loang **ngược** từ mỗi đại dương (đi sang ô có độ cao **không nhỏ hơn**), lấy giao.

In [24]:
def exercise_5_pacific_atlantic(heights: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_5_pacific_atlantic([[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]])) == [[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]
# assert exercise_5_pacific_atlantic([[1]]) == [[0, 0]]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_border` với các hàng tương ứng.

**Bài 3**
- Mức 1: loang từ ô đất trên biên, đánh dấu; đếm ô đất **chưa** đánh dấu.

**Bài 4**
- Mức 1: loang từ biên để "xóa" các đảo chạm biên (đổi `0` → `1`).
- Mức 2: rồi đếm số đảo còn lại như #200.

**Bài 5**
- Mức 1: hàm `reach(starts)` BFS từ danh sách ô, đi sang ô kề có `heights >= ` ô hiện tại.
- Mức 2: Thái Bình Dương: hàng 0 + cột 0; Đại Tây Dương: hàng `m−1` + cột `n−1`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
_ = trace_border(["OXXOX", "XOOXO", "XOXOX", "OXOOO", "XXOXO"])

Pha 1a — đánh dấu 'O' trên biên:
  # X X # X
  X O O X #
  X O X O X
  # X O O #
  X X # X #
Pha 1b — loang vào trong:
  # X X # X
  X O O X #
  X O X # X
  # X # # #
  X X # X #
Pha 2 — lật 'O' còn lại, khôi phục '#':
  O X X O X
  X X X X O
  X X X O X
  O X O O O
  X X O X O


## Đáp án Bài 2

In [26]:
def answer_2_solve(board: List[List[str]]) -> None:
    solve_border(board)

b = parse(["XXXX", "XOOX", "XXOX", "XOXX"]); answer_2_solve(b); assert ["".join(r) for r in b] == ["XXXX", "XXXX", "XXXX", "XOXX"]
b = parse(["X"]); answer_2_solve(b); assert b == [["X"]]
print("Bài 2 PASS (giống solve_border ở mục 5)")

Bài 2 PASS (giống solve_border ở mục 5)


## Đáp án Bài 3

In [27]:
def bfs_mark(grid, starts, target, mark):
    m, n = len(grid), len(grid[0])
    q = deque()
    for r, c in starts:
        if grid[r][c] == target:
            grid[r][c] = mark
            q.append((r, c))
    while q:
        x, y = q.popleft()
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == target:
                grid[nx][ny] = mark
                q.append((nx, ny))


def border_cells(m, n):
    return [(r, c) for r in range(m) for c in range(n) if r in (0, m - 1) or c in (0, n - 1)]


def answer_3_num_enclaves(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    bfs_mark(g, border_cells(len(g), len(g[0])), 1, 2)
    return sum(v == 1 for row in g for v in row)

assert answer_3_num_enclaves([[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]) == 3
assert answer_3_num_enclaves([[0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]]) == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_closed_island(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    bfs_mark(g, border_cells(m, n), 0, 1)              # xóa các đảo chạm biên
    count = 0
    for r in range(m):
        for c in range(n):
            if g[r][c] == 0:
                count += 1
                bfs_mark(g, [(r, c)], 0, 1)
    return count

assert answer_4_closed_island([[1,1,1,1,1,1,1,0],[1,0,0,0,0,1,1,0],[1,0,1,0,1,1,1,0],[1,0,0,0,0,1,0,1],[1,1,1,1,1,1,1,0]]) == 2
assert answer_4_closed_island([[0,0,1,0,0],[0,1,0,1,0],[0,1,1,1,0]]) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_pacific_atlantic(heights: List[List[int]]) -> List[List[int]]:
    m, n = len(heights), len(heights[0])
    def reach(starts):
        seen = set(starts)
        q = deque(starts)
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in seen and heights[nx][ny] >= heights[x][y]:
                    seen.add((nx, ny))
                    q.append((nx, ny))
        return seen
    pac = reach([(0, c) for c in range(n)] + [(r, 0) for r in range(1, m)])
    atl = reach([(m - 1, c) for c in range(n)] + [(r, n - 1) for r in range(m - 1)])
    return [list(p) for p in sorted(pac & atl)]

assert sorted(answer_5_pacific_atlantic([[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]])) == [[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]
assert answer_5_pacific_atlantic([[1]]) == [[0, 0]]
print("Bài 5 PASS")

Bài 5 PASS


Loang **ngược** chiều nước chảy (đi lên dốc) từ biên: tập ô đạt được chính là tập ô mà nước từ đó chảy được ra biên đó — cùng ý tưởng "đổi góc nhìn từ biên vào" của bài chính.

# 15. Tổng kết

> Đừng tìm vùng bị bao; hãy tìm vùng **thoát được**: BFS từ mọi `'O'` trên biên, đánh dấu `'#'`. Sau đó `'O' → 'X'`, `'#' → 'O'`. `O(m·n)`, dùng chính bảng để đánh dấu.

---

## Checklist trước khi nộp

- [ ] Loang từ **cả bốn** cạnh biên?
- [ ] Đánh dấu khi **đưa vào** hàng đợi?
- [ ] Khôi phục `'#'` → `'O'`?
- [ ] Dùng BFS/stack thay vì đệ quy sâu?